# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fasiuu/FLYRANK_TASK1/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
# Setup — shared feature builder from this repo + DuckDB connection to the hosted release.
# Token: Colab Secret HF_TOKEN (preferred) or a getpass prompt. Never paste a token into a cell.
%pip -q install duckdb
import os, sys, subprocess, json, warnings
warnings.filterwarnings('ignore')
if not os.path.exists('FLYRANK_TASK1'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://github.com/Fasiuu/FLYRANK_TASK1.git'], check=True)
sys.path.insert(0, 'FLYRANK_TASK1/work/scripts')
import numpy as np, pandas as pd
import flyrank_features as ff
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
np.random.seed(ff.SEED)
con = ff.connect()
os.makedirs('work/outputs', exist_ok=True)
print('connected; seed =', ff.SEED)

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The queue** for decision date 2026-06-30, the last day of the data, so July is the real future. The frozen recipe is retrained on every labelled snapshot and every client, then each page gets a risk score and a **rank inside its own site**. Ranking within a site matters because the label is site-relative, so a score only means something next to that site's other pages.

Every page also gets plain-language reason codes from the rule (what a reviewer will see first), an archetype, and one suggested action. **Archetype → action mapping:**

| archetype | what it looks like | action |
|---|---|---|
| **spike-propped** | last month leaned on one big day | **MONITOR**: confirm the spike before editing anything |
| **double slide** | impressions *and* rank both slipping | **REVIEW NOW**: compare with what ranks now, refresh |
| **quiet slide** | impressions slipping while rank holds | **REVIEW**: check query fit and freshness |
| **patchy** | visible on fewer than half the days | **MONITOR**: too thin to act on yet |
| **diffuse risk** | the model sees risk without one dominant sign | **REVIEW**: read the page against its main query |

In [ ]:
# Build the per-item panel for all decision dates incl. the sealed one and the deploy date (one scan).
import time
t0 = time.time()
panel = ff.build_panel(con, ff.ALL_DATES + [ff.DEPLOY_DATE])
panel['decision_date'] = pd.to_datetime(panel['decision_date']).dt.strftime('%Y-%m-%d')
m = ff.model_frame(panel)
lab = m[m.decision_date != ff.DEPLOY_DATE].copy()      # rows with an observed outcome month
dep = m[m.decision_date == ff.DEPLOY_DATE].copy()      # today's queue: the label month is the real future
print(f'{len(panel):,} item-date rows in {time.time()-t0:.0f}s; labelled rows {len(lab):,}; deploy rows {len(dep):,}')
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
MAKERS = {
    'logreg': lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    'hgb': lambda: HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                                  min_samples_leaf=100, random_state=ff.SEED),
    'rf': lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=ff.SEED),
}
FINAL = 'hgb'    # frozen in w05 on the development split (best top-of-queue precision), never re-chosen later
F = ff.FEATURES

def baseline(df):
    """Frozen transparent rule: one point per warning sign, ties broken by current visibility."""
    r = pd.DataFrame(index=df.index)
    r['sliding_impressions'] = df.imp_last30 < 0.85 * df.imp_prev30      # already losing impressions
    r['position_slipping']   = df.pos_change > 1.0                        # average rank got >1 place worse
    r['one_day_spike']       = df.spike_share > 0.25                      # month propped up by a single day
    r['aging_page']          = df.content_age_days > 270                  # past the decay cliff in FlyRank's paper
    r['patchy_visibility']   = df.days_with_imp_last30 < 15               # visible on fewer than half the days
    points = r.sum(axis=1)
    score = points + df.log_imp_last30 / 100.0
    reasons = r.apply(lambda row: '|'.join(c for c in r.columns if row[c]) or 'none', axis=1)
    return score, reasons, r

# Final model: the frozen recipe, now trained on every labelled snapshot and every client.
final = MAKERS[FINAL]().fit(lab[F], lab.declined.astype(int))
q = dep.copy()
q['risk_score'] = final.predict_proba(q[F])[:, 1]
q['risk_rank_in_site'] = q.groupby('client_hash_id').risk_score.rank(ascending=False, method='first').astype(int)
_, q['reason_codes'], _ = baseline(q)
def action(r):
    if r.spike_share > 0.25: return 'MONITOR: last month was propped up by a one-day spike - confirm before editing'
    if r.imp_momentum < -0.15 and r.pos_change > 1: return 'REVIEW NOW: impressions and rank both slipping - compare with the competing results, refresh'
    if r.imp_momentum < -0.15: return 'REVIEW: impressions slipping at a steady rank - check query fit and freshness'
    if r.days_with_imp_last30 < 15: return 'MONITOR: visibility too patchy to act on yet'
    return 'REVIEW: risk without one dominant sign - read the page against its main query'
q['action'] = q.apply(action, axis=1)
ARCH = {'MONITOR: last': 'spike-propped', 'REVIEW NOW': 'double slide', 'REVIEW: impressions': 'quiet slide',
        'MONITOR: visibility': 'patchy', 'REVIEW: risk': 'diffuse risk'}
q['archetype'] = q.action.map(lambda a: next(v for k, v in ARCH.items() if a.startswith(k)))
q['confidence'] = pd.qcut(q.risk_score, [0, .5, .8, 1.0], labels=['low', 'medium', 'high'])
q = q.sort_values(['client_hash_id', 'risk_rank_in_site'])
print(f'queue for decision date {ff.DEPLOY_DATE}: {len(q):,} pages from {q.client_hash_id.nunique()} sites')
show = ['client_hash_id', 'risk_rank_in_site', 'content_hash_id', 'risk_score', 'confidence', 'action', 'reason_codes']
print(q[q.risk_rank_in_site <= 5][show].head(25).round(3).to_string(index=False))
print("\narchetype -> action mapping, with how often each shows up in a site's top 10:")
t10 = q[q.risk_rank_in_site <= 10]
print(t10.groupby(['archetype', 'action']).size().rename('pages_in_top10').sort_values(ascending=False).reset_index().to_string(index=False))

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Who uses it:** a content or SEO reviewer at one site, once a month, working down that site's top 10-20 pages.

**What for:** deciding which pages to *open first*. Nothing more. The output is an order plus reasons, not a diagnosis.

**Where it stops being valid:**
- sites outside this kind of portfolio, or pages under 200 impressions per 60 days or younger than 90 days (they weren't in training),
- comparing scores *between* sites (the label is relative to each site),
- reading the score as a probability (it isn't calibrated; the tiers are percentiles),
- months with a big search-wide change unlike March-June 2026, until the monitoring below has been checked.

**Decay / refresh insight.** FlyRank's refresh logic leans on age ("refresh after ~270 days"). In this data, age is not the trigger: the oldest pages fall behind their site *least* often, while pages whose impressions are already falling fall behind most (tables below, with n). So the playbook schedules reviews by momentum and spikes, and treats age as context.

**Cost / value.** One review slot = a site's top 10 pages per month. The cell below turns the sealed-month precision into "real decliners found per site per month" for the model, the rule and random picking, with the review time stated as an assumption (30 minutes), not a measurement. A false alarm costs one wasted review; a miss costs weeks of lost visibility before a dashboard shows it.

In [ ]:
# Decay / refresh insight: is age the trigger, or momentum? (all labelled snapshots, n shown)
print('decline-vs-site rate by page age (all labelled snapshots):')
print(lab.groupby(pd.cut(lab.content_age_days, [90, 180, 270, 365, 100000], labels=['90-180d', '181-270d', '271-365d', '365d+']), observed=True)
         .declined.agg(['mean', 'size']).round(3).to_string())
print('\n...and by last-month momentum:')
print(lab.groupby(pd.cut(lab.imp_momentum, [-10, -0.3, -0.1, 0.1, 0.3, 10], labels=['falling fast', 'falling', 'flat', 'rising', 'rising fast']), observed=True)
         .declined.agg(['mean', 'size']).round(3).to_string())
# Cost / value: what one monthly review slot buys (review time is an ASSUMPTION, stated)
rp = 'FLYRANK_TASK1/work/outputs/sealed_metrics.json'
if os.path.exists(rp):
    sm = json.load(open(rp))['sealed']
    p_m, p_r, p_b = sm[FINAL]['client_p@10'], sm['baseline_rule']['client_p@10'], sm[FINAL]['client_base']
    n_sites, mins = q.client_hash_id.nunique(), 30
    print(f'\ncost/value: top-10 per site x {n_sites} sites = {10*n_sites} reviews a month (~{10*n_sites*mins/60:.0f} h at an assumed {mins} min each)')
    print(f'pages that really fall behind, per site per month (sealed June rates): model {10*p_m:.1f} | rule {10*p_r:.1f} | random {10*p_b:.1f}')
    print(f'-> same effort, about {10*(p_m-p_b):.1f} more real decliners caught per site than picking at random')
print('\nconfidence mix across the whole queue (tiers are score percentiles, not probabilities):')
print(q.confidence.value_counts().sort_index().to_string())
print('\npages per site in the queue (smallest 5):', q.client_hash_id.value_counts().tail(5).to_dict())

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any flagged page, a person checks:**
1. Is the drop real, or did last month spike? (look at the daily series)
2. Did a sibling page on the same site gain what this one lost? (consolidation, not decline)
3. Is it seasonal? (same month last year, where history exists)
4. Does the page still answer its main query better than what currently ranks?

**Never automated:** the no-go list below. The short version: nothing gets deleted, redirected or rewritten because a number is high.

In [ ]:
nogo = pd.DataFrame([
 ('delete / noindex / redirect a page', 'never automated - a fall can be a spike reverting, seasonality, or consolidation into a sibling page'),
 ('rewrite content because the score is high', 'a person reads the page and its query first; the score is a reason to look, not a diagnosis'),
 ('claim a refresh caused a recovery', 'needs a controlled before/after design; this queue is observational'),
 ('act on very small pages', 'population floor is 200 impressions / 60 days; below that it is noise'),
 ('compare scores across sites', 'the label is site-relative - rank WITHIN a site'),
], columns=['action', 'rule'])
print(nogo.to_string(index=False))

## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Signs the recommendations have gone stale:**
- **Input drift:** a population-stability index above 0.2 on any feature between the last labelled snapshot and the current one. The cell below computes it.
- **Label drift:** the site-relative decline rate leaving its ~48-52% band, or the median site ratio moving far from what training saw.
- **Outcome check:** every month, once the next month's data lands, score last month's queue: per-site precision@10 should stay well above that site's base rate. If it falls to the base rate for two months in a row, retrain or rethink.
- **Retrain trigger:** new month of data (monthly refit by default), any drift flag above, or a change to the GSC export.

In [ ]:
def psi(a, b, bins=10):
    qs = np.unique(np.quantile(a.dropna(), np.linspace(0, 1, bins + 1)))
    pa = np.histogram(a, qs)[0] / len(a) + 1e-6; pb = np.histogram(b, qs)[0] / len(b) + 1e-6
    return float(np.sum((pa - pb) * np.log(pa / pb)))
ref = lab[lab.decision_date == ff.SEALED_DATE]
drift = {f: round(psi(ref[f], dep[f]), 3) for f in F}
print('population stability index, last labelled snapshot -> deploy snapshot (> 0.2 = investigate):')
print(pd.Series(drift).sort_values(ascending=False).to_string())
print('\nsite-relative decline rate by labelled snapshot:', lab.groupby('decision_date').declined.mean().round(3).to_dict())
print('median site ratio by snapshot:', lab.groupby('decision_date').client_ratio.median().round(3).to_dict())

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The full queue goes to `work/outputs/action_queue.csv`. It's gitignored, because row-level data stays out of the public repo. The aggregate summary goes to `work/outputs/action_queue_summary.json`, which is committed and is what the paper's recommendations section builds on.

In [ ]:
export_cols = ['client_hash_id', 'risk_rank_in_site', 'content_hash_id', 'risk_score', 'confidence', 'archetype', 'action', 'reason_codes']
q[export_cols].to_csv('work/outputs/action_queue.csv', index=False)          # gitignored - stays local
summary = {'decision_date': ff.DEPLOY_DATE, 'pages': int(len(q)), 'sites': int(q.client_hash_id.nunique()),
           'top10_per_site_action_mix': q[q.risk_rank_in_site <= 10].action.str.split(':').str[0].value_counts().astype(int).to_dict(),
           'top10_archetypes': q[q.risk_rank_in_site <= 10].archetype.value_counts().astype(int).to_dict(),
           'top10_reason_codes': q[q.risk_rank_in_site <= 10].reason_codes.str.split('|').explode().value_counts().astype(int).to_dict(),
           'drift_psi': drift}
json.dump(summary, open('work/outputs/action_queue_summary.json', 'w'), indent=2, default=str)
print('wrote work/outputs/action_queue.csv (local only) and work/outputs/action_queue_summary.json')
print('SUMMARY_JSON ' + json.dumps(summary, default=str))

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.